# 0. Configuración

In [1]:
# ============================================================
# DETECCIÓN AUTOMÁTICA DATASET KAGGLE
# ============================================================

from pathlib import Path
import shutil

INPUT_ROOT = Path("/kaggle/input")

WORK = Path("/kaggle/working")

candidate_roots = []

for p in INPUT_ROOT.rglob("*"):

    if p.is_dir():

        exp_folders = [
            p / "exp01_local",
            p / "exp02_29f",
            p / "exp03_33f"
        ]

        if all(x.exists() for x in exp_folders):

            candidate_roots.append(p)

if len(candidate_roots) == 0:

    raise ValueError(
        "No se encontró dataset válido."
    )

INPUT_BASE = candidate_roots[0]

print("Dataset detectado:")
print(INPUT_BASE)



Dataset detectado:
/kaggle/input/datasets/sity127uoc/m5-uoc-powerbi-etl


In [2]:
# ============================================================
# ESTRUCTURA
# ============================================================

PROJECT_ROOT = WORK / "TFM"

POWERBI = PROJECT_ROOT / "powerbi"

POWERBI_LAKE = POWERBI / "data" / "powerbi_lake"

DIM_DIR = POWERBI_LAKE / "dims"

FACT_DIR = POWERBI_LAKE / "facts"

TOOLS_DIR = POWERBI / "tools"

DUCKDB_TOOLS = TOOLS_DIR / "duckdb"

ODBC_TOOLS = TOOLS_DIR / "duckdb_odbc"

# ============================================================
# CREAR CARPETAS
# ============================================================

for p in [
    DIM_DIR,
    FACT_DIR,
    TOOLS_DIR,
    DUCKDB_TOOLS,
    ODBC_TOOLS
]:

    p.mkdir(
        parents=True,
        exist_ok=True
    )

# ============================================================
# CONFIGURACIÓN EXPERIMENTOS
# ============================================================

EXPERIMENTS = {

    "exp01_local": {
        "path": INPUT_BASE / "exp01_local",
        "experiment_name": "Local baseline",
        "model_name": "LightGBM global incremental",
        "features_used": 25,
        "environment": "local_i5_32gb"
    },

    "exp02_29f": {
        "path": INPUT_BASE / "exp02_29f",
        "experiment_name": "Kaggle 29 features",
        "model_name": "LightGBM global incremental",
        "features_used": 29,
        "environment": "kaggle_cpu"
    },

    "exp03_33f": {
        "path": INPUT_BASE / "exp03_33f",
        "experiment_name": "Kaggle 33 features",
        "model_name": "LightGBM global incremental",
        "features_used": 33,
        "environment": "kaggle_cpu"
    }
}

## 0.1 Copiar los ficheros de experimentos a Working.kaggle

In [3]:
# ============================================================
# COPY INPUT EXPERIMENTS
# ============================================================

WORK_EXPERIMENTS = WORK / "experiments"

WORK_EXPERIMENTS.mkdir(
    parents=True,
    exist_ok=True
)

for exp_id, meta in EXPERIMENTS.items():

    src = meta["path"]

    dst = WORK_EXPERIMENTS / exp_id

    if dst.exists():

        shutil.rmtree(dst)

    shutil.copytree(
        src,
        dst
    )

    EXPERIMENTS[exp_id]["path"] = dst

    print(f"\nCopied {exp_id}")

    print(f"From: {src}")

    print(f"To:   {dst}")




Copied exp01_local
From: /kaggle/input/datasets/sity127uoc/m5-uoc-powerbi-etl/exp01_local
To:   /kaggle/working/experiments/exp01_local

Copied exp02_29f
From: /kaggle/input/datasets/sity127uoc/m5-uoc-powerbi-etl/exp02_29f
To:   /kaggle/working/experiments/exp02_29f

Copied exp03_33f
From: /kaggle/input/datasets/sity127uoc/m5-uoc-powerbi-etl/exp03_33f
To:   /kaggle/working/experiments/exp03_33f


In [4]:
# ============================================================
# COPY SQL SCRIPT
# ============================================================

sql_src = INPUT_BASE / "create_powerbi_model.sql"

sql_dst = TOOLS_DIR / "create_powerbi_model.sql"

shutil.copy(
    sql_src,
    sql_dst
)

print("\nSQL copied:")

print(sql_dst)

# ============================================================
# VALIDACIÓN FINAL
# ============================================================

print("\n======================================")

print("Estructura preparada correctamente.")

print("\nPower BI lake:")

print(POWERBI_LAKE)

print("\nDims:")

print(DIM_DIR)

print("\nFacts:")

print(FACT_DIR)

print("\nDuckDB tools:")

print(DUCKDB_TOOLS)

print("\nODBC tools:")

print(ODBC_TOOLS)


SQL copied:
/kaggle/working/TFM/powerbi/tools/create_powerbi_model.sql

Estructura preparada correctamente.

Power BI lake:
/kaggle/working/TFM/powerbi/data/powerbi_lake

Dims:
/kaggle/working/TFM/powerbi/data/powerbi_lake/dims

Facts:
/kaggle/working/TFM/powerbi/data/powerbi_lake/facts

DuckDB tools:
/kaggle/working/TFM/powerbi/tools/duckdb

ODBC tools:
/kaggle/working/TFM/powerbi/tools/duckdb_odbc


# 1. Verificación de inputs

In [5]:
required_files = [
    "hierarchy_predictions.parquet",
    "mint_predictions.parquet",
    "wrmsse_details.parquet",
    "wrmsse_result.parquet",
    "shap_values.parquet",
    "weights.parquet"
]

optional_files = [
    "scale.parquet",
    "mint_reconciled.parquet"
]

for exp_id, meta in EXPERIMENTS.items():

    exp_path = meta["path"]

    print("\n" + "=" * 60)
    print(f"Checking experiment: {exp_id}")
    print(f"Path: {exp_path}")

    if not exp_path.exists():
        print("ERROR: folder not found")
        continue

    files = sorted([x.name for x in exp_path.glob("*")])

    print(f"Files found: {len(files)}")

    missing_required = []

    for f in required_files:
        if (exp_path / f).exists():
            print(f"OK   {f}")
        else:
            print(f"MISS {f}")
            missing_required.append(f)

    print("\nOptional files:")

    for f in optional_files:
        if (exp_path / f).exists():
            print(f"OK   {f}")
        else:
            print(f"SKIP {f}")

    pred_files = sorted(exp_path.glob("pred_*.parquet"))

    print(f"\nPrediction partitions: {len(pred_files)}")

    for p in pred_files:
        print(" -", p.name)

    if len(missing_required) == 0:
        print("\nSTATUS: READY")
    else:
        print("\nSTATUS: INCOMPLETE")
        print("Missing:", missing_required)


Checking experiment: exp01_local
Path: /kaggle/working/experiments/exp01_local
Files found: 6
OK   hierarchy_predictions.parquet
OK   mint_predictions.parquet
OK   wrmsse_details.parquet
OK   wrmsse_result.parquet
OK   shap_values.parquet
OK   weights.parquet

Optional files:
SKIP scale.parquet
SKIP mint_reconciled.parquet

Prediction partitions: 0

STATUS: READY

Checking experiment: exp02_29f
Path: /kaggle/working/experiments/exp02_29f
Files found: 6
OK   hierarchy_predictions.parquet
OK   mint_predictions.parquet
OK   wrmsse_details.parquet
OK   wrmsse_result.parquet
OK   shap_values.parquet
OK   weights.parquet

Optional files:
SKIP scale.parquet
SKIP mint_reconciled.parquet

Prediction partitions: 0

STATUS: READY

Checking experiment: exp03_33f
Path: /kaggle/working/experiments/exp03_33f
Files found: 6
OK   hierarchy_predictions.parquet
OK   mint_predictions.parquet
OK   wrmsse_details.parquet
OK   wrmsse_result.parquet
OK   shap_values.parquet
OK   weights.parquet

Optional fil

# 2. Verificación de coherencia de fichero de mint_predictions.parquet + función de fallback (reconstruir predicciones faltantes en mint_predictions con las de bottom_up, si es necesario)
En algunos experimentos locales la reconciliación MinT exacta completa no pudo ejecutarse por limitaciones de memoria, por lo que se aplicaron versiones parciales/muestrales que generaron predicciones reconciliadas artificialmente bajas. Para mantener comparabilidad entre experimentos, aquí se verifica la escala de mint_predictions.parquet y, si se detectan reconciliaciones parciales o valores anómalamente bajos, las series no reconciliadas correctamente se rellenan utilizando las predicciones Bottom-Up originales provenientes de hierarchy_predictions.parquet.

In [6]:
# ============================================================
# VALIDACIÓN Y CORRECCIÓN OPCIONAL DE MINT POR EXPERIMENTO
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np

AUTO_FIX_MINT = False

# ============================================================
# REGENERAR MINT COMO BOTTOM-UP
# ============================================================

def regenerate_mint_as_bottom_up(exp_path):

    exp_path = Path(exp_path)

    hierarchy = pd.read_parquet(
        exp_path / "hierarchy_predictions.parquet",
        columns=["series_id", "date", "prediction"]
    )

    hierarchy = hierarchy.rename(
        columns={"prediction": "prediction_mint"}
    )

    hierarchy.to_parquet(
        exp_path / "mint_predictions.parquet",
        engine="pyarrow",
        coerce_timestamps="ms",
        allow_truncated_timestamps=True,
        index=False
    )

    print(
        f"FIXED: {exp_path.name} "
        f"-> mint regenerated as Bottom-Up fallback"
    )

# ============================================================
# CHECK MINT QUALITY
# ============================================================

def check_mint_quality_fast(exp_path):

    exp_path = Path(exp_path)

    h = pd.read_parquet(
        exp_path / "hierarchy_predictions.parquet",
        columns=["prediction"]
    )

    mint = pd.read_parquet(
        exp_path / "mint_predictions.parquet",
        columns=["prediction_mint"]
    )

    base_sum = h["prediction"].sum()

    mint_sum = mint["prediction_mint"].sum()

    ratio = (
        mint_sum / base_sum
        if base_sum != 0
        else np.nan
    )

    nulls = mint["prediction_mint"].isna().sum()

    bad_scale = not (0.8 <= ratio <= 1.2)

    has_nulls = nulls > 0

    print("\n" + "=" * 60)

    print(f"Checking mint quality: {exp_path.name}")

    print(f"Base sum: {base_sum}")

    print(f"Mint sum: {mint_sum}")

    print(f"Ratio: {ratio}")

    print(f"Nulls: {nulls}")

    if bad_scale or has_nulls:
        print("STATUS: WARNING")
    else:
        print("STATUS: OK")

    return bad_scale, has_nulls

# ============================================================
# VALIDACIÓN GLOBAL
# ============================================================

for exp_id, cfg in EXPERIMENTS.items():

    exp_path = cfg["path"]

    bad_scale, has_nulls = check_mint_quality_fast(
        exp_path
    )

    if bad_scale or has_nulls:

        print(f"{exp_id}: mint requires review")

        if AUTO_FIX_MINT:

            print(
                f"{exp_id}: "
                f"regenerating mint as Bottom-Up fallback"
            )

            regenerate_mint_as_bottom_up(
                exp_path
            )

            check_mint_quality_fast(
                exp_path
            )

        else:

            print(
                f"{exp_id}: "
                f"AUTO_FIX_MINT is False, no changes applied"
            )

    else:

        print(f"{exp_id}: no fix needed")


Checking mint quality: exp01_local
Base sum: 514283939.3830852
Mint sum: 352.0559072044477
Ratio: 6.845555154352285e-07
Nulls: 0
STATUS: WARNING
exp01_local: mint requires review
exp01_local: AUTO_FIX_MINT is False, no changes applied

Checking mint quality: exp02_29f
Base sum: 509045179.8636107
Mint sum: 247916411.51466995
Ratio: 0.48702241239391486
Nulls: 0
STATUS: WARNING
exp02_29f: mint requires review
exp02_29f: AUTO_FIX_MINT is False, no changes applied

Checking mint quality: exp03_33f
Base sum: 507140699.0062593
Mint sum: 246533720.61908656
Ratio: 0.4861248980848286
Nulls: 0
STATUS: WARNING
exp03_33f: mint requires review
exp03_33f: AUTO_FIX_MINT is False, no changes applied


# 4. build_dimensions

## 4.1 dim_date

In [7]:
# =========================
# DIM_DATE
# =========================

reference_exp = EXPERIMENTS["exp01_local"]["path"]

hierarchy = pd.read_parquet(
    reference_exp / "hierarchy_predictions.parquet",
    columns=["date"]
)

dim_date = (
    hierarchy[["date"]]
    .drop_duplicates()
    .sort_values("date")
    .reset_index(drop=True)
)

dim_date["date"] = pd.to_datetime(dim_date["date"])

dim_date["date_key"] = (
    dim_date["date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = dim_date["date"].dt.quarter
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name()

dim_date["week"] = (
    dim_date["date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

dim_date["day"] = dim_date["date"].dt.day
dim_date["weekday"] = dim_date["date"].dt.day_name()

dim_date["weekday_num"] = (
    dim_date["date"].dt.weekday + 1
)

dim_date["is_weekend"] = (
    dim_date["weekday_num"].isin([6, 7])
)

dim_date.to_parquet(
    DIM_DIR / "dim_date.parquet",
    index=False
)

print("dim_date saved")
print(dim_date.shape)

dim_date.head()

dim_date saved
(1913, 11)


,date,date_key,year,quarter,month,month_name,week,day,weekday,weekday_num,is_weekend
0,2011-01-29,20110129,2011,1,1,January,4,29,Saturday,6,True
1,2011-01-30,20110130,2011,1,1,January,4,30,Sunday,7,True
2,2011-01-31,20110131,2011,1,1,January,5,31,Monday,1,False
3,2011-02-01,20110201,2011,1,2,February,5,1,Tuesday,2,False
4,2011-02-02,20110202,2011,1,2,February,5,2,Wednesday,3,False


## 4.2 dim_experiment

In [8]:
# =========================
# DIM_EXPERIMENT
# =========================

rows = []

for exp_id, cfg in EXPERIMENTS.items():

    exp_path = cfg["path"]

    wrmsse_path = exp_path / "wrmsse_result.parquet"

    if wrmsse_path.exists():

        wrmsse_df = pd.read_parquet(wrmsse_path)

        wrmsse_value = wrmsse_df.loc[
            wrmsse_df["metric"] == "WRMSSE",
            "value"
        ].iloc[0]

    else:
        wrmsse_value = np.nan

    rows.append({
        "experiment_id": exp_id,
        "experiment_name": cfg["experiment_name"],
        "environment": cfg["environment"],
        "model": cfg["model_name"],
        "features_used": cfg["features_used"],
        "mint_method": "Bottom-Up fallback",
        "wrmsse": wrmsse_value
    })

dim_experiment = pd.DataFrame(rows)

dim_experiment.to_parquet(
    DIM_DIR / "dim_experiment.parquet",
    index=False
)

print("dim_experiment saved")
print(dim_experiment.shape)

dim_experiment

dim_experiment saved
(3, 7)


,experiment_id,experiment_name,environment,model,features_used,mint_method,wrmsse
0,exp01_local,Local baseline,local_i5_32gb,LightGBM global incremental,25,Bottom-Up fallback,1.302129
1,exp02_29f,Kaggle 29 features,kaggle_cpu,LightGBM global incremental,29,Bottom-Up fallback,0.768289
2,exp03_33f,Kaggle 33 features,kaggle_cpu,LightGBM global incremental,33,Bottom-Up fallback,0.767451


## 4.3 dim_level

In [9]:
# =========================
# DIM_LEVEL
# =========================

dim_level = pd.DataFrame({
    "level": [
        "item",
        "dept",
        "cat",
        "store",
        "state",
        "state_cat",
        "state_dept",
        "total"
    ],
    "level_order": [1, 2, 3, 4, 5, 6, 7, 8]
})

dim_level.to_parquet(
    DIM_DIR / "dim_level.parquet",
    index=False
)

print("dim_level saved")
print(dim_level.shape)

dim_level

dim_level saved
(8, 2)


,level,level_order
0,item,1
1,dept,2
2,cat,3
3,store,4
4,state,5
5,state_cat,6
6,state_dept,7
7,total,8


## 4.4 dim_feature

In [10]:
# =========================
# DIM_FEATURE
# =========================

base_features = [
    "item_id",
    "wm_yr_wk",
    "dept_id",
    "cat_id",
    "state_id",
    "weekday",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA",
    "snap_TX",
    "snap_WI",
    "sell_price",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28",
    "rolling_max_28",
    "rolling_min_28",
    "price_lag_1",
    "price_diff_1",
    "price_pct_change_1",
    "price_mean_item_store",
    "price_norm_item_store",
    "weekofyear"
]

dim_feature = pd.DataFrame({
    "feature": base_features,
    "feature_order": range(1, len(base_features) + 1)
})

dim_feature["feature_group"] = np.select(
    [
        dim_feature["feature"].str.startswith("lag_"),
        dim_feature["feature"].str.startswith("rolling_"),
        dim_feature["feature"].str.startswith("price_"),
        dim_feature["feature"].str.startswith("event_"),
        dim_feature["feature"].str.startswith("snap_"),
        dim_feature["feature"].isin(["weekday", "wday", "month", "year", "weekofyear", "wm_yr_wk"]),
        dim_feature["feature"].isin(["item_id", "dept_id", "cat_id", "state_id"])
    ],
    [
        "Lag features",
        "Rolling features",
        "Price features",
        "Event features",
        "SNAP features",
        "Calendar features",
        "Categorical identifiers"
    ],
    default="Other"
)

dim_feature.to_parquet(
    DIM_DIR / "dim_feature.parquet",
    index=False
)

print("dim_feature saved")
print(dim_feature.shape)

dim_feature

dim_feature saved
(34, 3)


,feature,feature_order,feature_group
0,item_id,1,Categorical identifiers
1,wm_yr_wk,2,Calendar features
2,dept_id,3,Categorical identifiers
3,cat_id,4,Categorical identifiers
4,state_id,5,Categorical identifiers
5,weekday,6,Calendar features
6,wday,7,Calendar features
7,month,8,Calendar features
8,year,9,Calendar features
9,event_name_1,10,Event features


## 4.5 dim_series

In [11]:
# =========================
# DIM_SERIES
# =========================

import pandas as pd

reference_exp = EXPERIMENTS["exp01_local"]["path"]

hierarchy = pd.read_parquet(
    reference_exp / "hierarchy_predictions.parquet",
    columns=["series_id", "level"]
).drop_duplicates()

dim_series = hierarchy.copy()

dim_series["series_id"] = dim_series["series_id"].astype("string")
dim_series["level"] = dim_series["level"].astype("string")

for col in ["store_id", "state_id", "cat_id", "dept_id", "item_id"]:
    dim_series[col] = pd.Series([pd.NA] * len(dim_series), dtype="string")

# item: CA_1_FOODS_1_001
mask = dim_series["level"] == "item"
parts = dim_series.loc[mask, "series_id"].str.split("_")

dim_series.loc[mask, "store_id"] = parts.str[0] + "_" + parts.str[1]
dim_series.loc[mask, "state_id"] = dim_series.loc[mask, "store_id"].str[:2]
dim_series.loc[mask, "cat_id"] = parts.str[2]
dim_series.loc[mask, "dept_id"] = parts.str[2] + "_" + parts.str[3]
dim_series.loc[mask, "item_id"] = parts.str[2] + "_" + parts.str[3] + "_" + parts.str[4]

# store: CA_1
mask = dim_series["level"] == "store"
dim_series.loc[mask, "store_id"] = dim_series.loc[mask, "series_id"]
dim_series.loc[mask, "state_id"] = dim_series.loc[mask, "store_id"].str[:2]

# state: CA
mask = dim_series["level"] == "state"
dim_series.loc[mask, "state_id"] = dim_series.loc[mask, "series_id"]

# cat: CA_1_FOODS
mask = dim_series["level"] == "cat"
parts = dim_series.loc[mask, "series_id"].str.split("_")
dim_series.loc[mask, "store_id"] = parts.str[0] + "_" + parts.str[1]
dim_series.loc[mask, "state_id"] = dim_series.loc[mask, "store_id"].str[:2]
dim_series.loc[mask, "cat_id"] = parts.str[2]

# dept: CA_1_FOODS_1
mask = dim_series["level"] == "dept"
parts = dim_series.loc[mask, "series_id"].str.split("_")
dim_series.loc[mask, "store_id"] = parts.str[0] + "_" + parts.str[1]
dim_series.loc[mask, "state_id"] = dim_series.loc[mask, "store_id"].str[:2]
dim_series.loc[mask, "cat_id"] = parts.str[2]
dim_series.loc[mask, "dept_id"] = parts.str[2] + "_" + parts.str[3]

# state_cat: CA_FOODS
mask = dim_series["level"] == "state_cat"
parts = dim_series.loc[mask, "series_id"].str.split("_")
dim_series.loc[mask, "state_id"] = parts.str[0]
dim_series.loc[mask, "cat_id"] = parts.str[1]

# state_dept: CA_FOODS_1
mask = dim_series["level"] == "state_dept"
parts = dim_series.loc[mask, "series_id"].str.split("_")
dim_series.loc[mask, "state_id"] = parts.str[0]
dim_series.loc[mask, "cat_id"] = parts.str[1]
dim_series.loc[mask, "dept_id"] = parts.str[1] + "_" + parts.str[2]

# total
mask = dim_series["level"] == "total"
dim_series.loc[mask, "series_id"] = "total"

dim_series = dim_series.drop_duplicates("series_id").reset_index(drop=True)

for col in [
    "store_id",
    "state_id",
    "cat_id",
    "dept_id",
    "item_id"
]:
    dim_series[col] = dim_series[col].fillna("Other")


dim_series.to_parquet(
    DIM_DIR / "dim_series.parquet",
    index=False
)

print("dim_series saved")
print(dim_series.shape)

dim_series.head()

dim_series saved
(30634, 7)


,series_id,level,store_id,state_id,cat_id,dept_id,item_id
0,CA_1_FOODS_1_001,item,CA_1,CA,FOODS,FOODS_1,FOODS_1_001
1,CA_1_FOODS_1_002,item,CA_1,CA,FOODS,FOODS_1,FOODS_1_002
2,CA_1_FOODS_1_003,item,CA_1,CA,FOODS,FOODS_1,FOODS_1_003
3,CA_1_FOODS_1_004,item,CA_1,CA,FOODS,FOODS_1,FOODS_1_004
4,CA_1_FOODS_1_005,item,CA_1,CA,FOODS,FOODS_1,FOODS_1_005


# 5.Construcción de tablas fact
Se construyen tablas fact consolidadas anexando los distintos experimentos para permitir comparativa unificada de métricas, predicciones y rendimiento entre modelos.

## 5.1 fact_forecast agg

In [12]:
# =========================
# FACT_FORECAST_AGG
# =========================

import gc

OUT = FACT_DIR / "fact_forecast_agg"

if OUT.exists():
    shutil.rmtree(OUT)

levels_keep = [
    "total",
    "state",
    "store",
    "cat",
    "dept",
    "state_cat",
    "state_dept"
]

for exp_id, cfg in EXPERIMENTS.items():

    exp_path = cfg["path"]
    model_name = cfg["model_name"]

    print("\n" + "=" * 60)
    print(f"Processing experiment: {exp_id}")

    for level in levels_keep:

        print("Processing level:", level)

        hierarchy_part = pd.read_parquet(
            exp_path / "hierarchy_predictions.parquet",
            columns=["series_id", "date", "prediction", "sales", "level"],
            filters=[("level", "==", level)]
        )

        # Some local outputs contain duplicated rows for the same
        # series_id/date/level. Mean removes duplication without inflating sales.
        hierarchy_part = (
            hierarchy_part
            .groupby(
                ["series_id", "date", "level"],
                as_index=False,
                observed=False
            )
            .agg(
                prediction=("prediction", "mean"),
                sales=("sales", "mean")
            )
        )

        series_list = (
            hierarchy_part["series_id"]
            .drop_duplicates()
            .tolist()
        )

        mint_part = pd.read_parquet(
            exp_path / "mint_predictions.parquet",
            columns=["series_id", "date", "prediction_mint"],
            filters=[("series_id", "in", series_list)]
        )

        mint_part = (
            mint_part
            .groupby(
                ["series_id", "date"],
                as_index=False,
                observed=False
            )
            .agg(
                prediction_mint=("prediction_mint", "mean")
            )
        )

        fact_part = hierarchy_part.merge(
            mint_part,
            on=["series_id", "date"],
            how="left"
        )

        fact_part["date"] = pd.to_datetime(fact_part["date"])
        fact_part["date_key"] = (
            fact_part["date"]
            .dt.strftime("%Y%m%d")
            .astype(int)
        )

        fact_part = fact_part.rename(columns={
            "sales": "sales_real",
            "prediction": "prediction_base"
        })

        fact_part["prediction_mint"] = fact_part["prediction_mint"].fillna(
            fact_part["prediction_base"]
        )

        fact_part["experiment_id"] = exp_id
        fact_part["model_name"] = model_name

        fact_part["error"] = (
            fact_part["sales_real"]
            - fact_part["prediction_mint"]
        )

        fact_part["abs_error"] = fact_part["error"].abs()
        fact_part["sq_error"] = fact_part["error"] ** 2

        fact_part["ape"] = np.where(
            fact_part["sales_real"] != 0,
            fact_part["abs_error"] / fact_part["sales_real"],
            np.nan
        )

        fact_part = fact_part[
            [
                "date_key",
                "date",
                "series_id",
                "level",
                "experiment_id",
                "model_name",
                "sales_real",
                "prediction_base",
                "prediction_mint",
                "error",
                "abs_error",
                "sq_error",
                "ape"
            ]
        ]

        fact_part.to_parquet(
            OUT,
            index=False,
            partition_cols=["experiment_id", "level"],
            engine="pyarrow"
        )

        print(exp_id, level, fact_part.shape)

        del hierarchy_part, mint_part, fact_part
        gc.collect()

print("Saved:", OUT)


Processing experiment: exp01_local
Processing level: total
exp01_local total (1913, 13)
Processing level: state
exp01_local state (5739, 13)
Processing level: store
exp01_local store (19130, 13)
Processing level: cat
exp01_local cat (57390, 13)
Processing level: dept
exp01_local dept (133910, 13)
Processing level: state_cat
exp01_local state_cat (17217, 13)
Processing level: state_dept
exp01_local state_dept (40173, 13)

Processing experiment: exp02_29f
Processing level: total
exp02_29f total (1913, 13)
Processing level: state
exp02_29f state (5739, 13)
Processing level: store
exp02_29f store (19130, 13)
Processing level: cat
exp02_29f cat (57390, 13)
Processing level: dept
exp02_29f dept (133910, 13)
Processing level: state_cat
exp02_29f state_cat (17217, 13)
Processing level: state_dept
exp02_29f state_dept (40173, 13)

Processing experiment: exp03_33f
Processing level: total
exp03_33f total (1913, 13)
Processing level: state
exp03_33f state (5739, 13)
Processing level: store
exp03_

# 5.2 fact_forecast_monthly

In [13]:
# =========================
# FACT_FORECAST_MONTHLY
# =========================

forecast_agg = pd.read_parquet(
    FACT_DIR / "fact_forecast_agg"
)

forecast_agg["date"] = pd.to_datetime(forecast_agg["date"])
forecast_agg["year_month"] = forecast_agg["date"].dt.strftime("%Y-%m")

fact_forecast_monthly = (
    forecast_agg
    .groupby(
        ["experiment_id", "level", "year_month"],
        as_index=False,
        observed=False
    )
    .agg(
        sales_real=("sales_real", "sum"),
        prediction_base=("prediction_base", "sum"),
        prediction_mint=("prediction_mint", "sum"),
        abs_error=("abs_error", "sum"),
        sq_error=("sq_error", "sum")
    )
)

fact_forecast_monthly["error"] = (
    fact_forecast_monthly["sales_real"]
    - fact_forecast_monthly["prediction_mint"]
)

fact_forecast_monthly["ape"] = np.where(
    fact_forecast_monthly["sales_real"] != 0,
    fact_forecast_monthly["abs_error"] / fact_forecast_monthly["sales_real"],
    np.nan
)

fact_forecast_monthly.to_parquet(
    FACT_DIR / "fact_forecast_monthly.parquet",
    index=False
)

print("fact_forecast_monthly saved")
print(fact_forecast_monthly.shape)

fact_forecast_monthly.head()

fact_forecast_monthly saved
(1344, 10)


,experiment_id,level,year_month,sales_real,prediction_base,prediction_mint,abs_error,sq_error,error,ape
0,exp01_local,cat,2011-01,88163.0,86598.796533,0.000005,88162.999995,1.624229e+08,88162.999995,1.0
1,exp01_local,cat,2011-02,726375.0,722152.913649,0.000037,726374.999963,1.250640e+09,726374.999963,1.0
2,exp01_local,cat,2011-03,763567.0,772885.130794,0.000039,763566.999961,1.189524e+09,763566.999961,1.0
3,exp01_local,cat,2011-04,737713.0,747103.808751,0.000041,737712.999959,1.124307e+09,737712.999959,1.0
4,exp01_local,cat,2011-05,719562.0,724524.497385,0.000041,719561.999959,1.002194e+09,719561.999959,1.0


# 5.3 fact_top_items usando batches

In [14]:
# =========================
# FACT_TOP_ITEMS
# =========================

import gc
import pyarrow.dataset as ds

TOP_N = 1000

top_items_parts = []

for exp_id, cfg in EXPERIMENTS.items():

    exp_path = cfg["path"]
    model_name = cfg["model_name"]

    print("\n" + "=" * 60)
    print(f"Processing top items: {exp_id}")

    wrmsse_details = pd.read_parquet(
        exp_path / "wrmsse_details.parquet"
    )

    top_items = (
        wrmsse_details
        .sort_values("wrmsse_component", ascending=False)
        .head(TOP_N)["series_id"]
        .tolist()
    )

    hierarchy_ds = ds.dataset(
        exp_path / "hierarchy_predictions.parquet",
        format="parquet"
    )

    scanner = hierarchy_ds.scanner(
        columns=["series_id", "date", "prediction", "sales", "level"],
        filter=(
            ds.field("level") == "item"
        ) & (
            ds.field("series_id").isin(top_items)
        ),
        batch_size=100_000
    )

    hierarchy_items = scanner.to_table().to_pandas()

    mint_ds = ds.dataset(
        exp_path / "mint_predictions.parquet",
        format="parquet"
    )

    scanner = mint_ds.scanner(
        columns=["series_id", "date", "prediction_mint"],
        filter=ds.field("series_id").isin(top_items),
        batch_size=100_000
    )

    mint_items = scanner.to_table().to_pandas()

    fact_part = hierarchy_items.merge(
        mint_items,
        on=["series_id", "date"],
        how="left"
    )

    fact_part["date"] = pd.to_datetime(fact_part["date"])
    fact_part["date_key"] = fact_part["date"].dt.strftime("%Y%m%d").astype(int)

    fact_part = fact_part.rename(columns={
        "sales": "sales_real",
        "prediction": "prediction_base"
    })

    fact_part["prediction_mint"] = fact_part["prediction_mint"].fillna(
        fact_part["prediction_base"]
    )

    fact_part["experiment_id"] = exp_id
    fact_part["model_name"] = model_name

    fact_part["error"] = fact_part["sales_real"] - fact_part["prediction_mint"]
    fact_part["abs_error"] = fact_part["error"].abs()
    fact_part["sq_error"] = fact_part["error"] ** 2

    fact_part["ape"] = np.where(
        fact_part["sales_real"] != 0,
        fact_part["abs_error"] / fact_part["sales_real"],
        np.nan
    )

    parts = fact_part["series_id"].str.split("_", expand=True)

    fact_part["store_id"] = parts[0] + "_" + parts[1]
    fact_part["cat_id"] = parts[2]
    fact_part["dept_id"] = parts[2] + "_" + parts[3]
    fact_part["item_code"] = parts[2] + "_" + parts[3] + "_" + parts[4]

    fact_part = fact_part[
        [
            "date_key",
            "date",
            "series_id",
            "store_id",
            "cat_id",
            "dept_id",
            "item_code",
            "level",
            "experiment_id",
            "model_name",
            "sales_real",
            "prediction_base",
            "prediction_mint",
            "error",
            "abs_error",
            "sq_error",
            "ape"
        ]
    ]

    top_items_parts.append(fact_part)

    print(exp_id, fact_part.shape)

    del wrmsse_details, hierarchy_items, mint_items, fact_part
    gc.collect()

fact_top_items = pd.concat(
    top_items_parts,
    ignore_index=True
)

fact_top_items.to_parquet(
    FACT_DIR / "fact_top_items.parquet",
    index=False
)

print("fact_top_items saved")
print(fact_top_items.shape)

fact_top_items.head()


Processing top items: exp01_local
exp01_local (1913000, 17)

Processing top items: exp02_29f
exp02_29f (1913000, 17)

Processing top items: exp03_33f
exp03_33f (1913000, 17)
fact_top_items saved
(5739000, 17)


,date_key,date,series_id,store_id,cat_id,dept_id,item_code,level,experiment_id,model_name,sales_real,prediction_base,prediction_mint,error,abs_error,sq_error,ape
0,20110129,2011-01-29,CA_1_FOODS_1_004,CA_1,FOODS,FOODS_1,FOODS_1_004,item,exp01_local,LightGBM global incremental,0,5.234208,1.772939e-07,-1.772939e-07,1.772939e-07,3.143312e-14,NaN
1,20110130,2011-01-30,CA_1_FOODS_1_004,CA_1,FOODS,FOODS_1,FOODS_1_004,item,exp01_local,LightGBM global incremental,0,0.670901,2.272486e-08,-2.272486e-08,2.272486e-08,5.164193e-16,NaN
2,20110131,2011-01-31,CA_1_FOODS_1_004,CA_1,FOODS,FOODS_1,FOODS_1_004,item,exp01_local,LightGBM global incremental,0,0.132146,4.476065e-09,-4.476065e-09,4.476065e-09,2.003515e-17,NaN
3,20110201,2011-02-01,CA_1_FOODS_1_004,CA_1,FOODS,FOODS_1,FOODS_1_004,item,exp01_local,LightGBM global incremental,0,0.042490,1.439244e-09,-1.439244e-09,1.439244e-09,2.071424e-18,NaN
4,20110202,2011-02-02,CA_1_FOODS_1_004,CA_1,FOODS,FOODS_1,FOODS_1_004,item,exp01_local,LightGBM global incremental,0,0.391549,1.326260e-08,-1.326260e-08,1.326260e-08,1.758965e-16,NaN


## 5.4 fact_wrmsse

In [15]:
# =========================
# FACT_WRMSSE
# =========================

wrmsse_parts = []

for exp_id, cfg in EXPERIMENTS.items():

    exp_path = cfg["path"]

    wrmsse_details = pd.read_parquet(
        exp_path / "wrmsse_details.parquet"
    )

    fact_part = wrmsse_details.copy()
    fact_part["experiment_id"] = exp_id

    fact_part = fact_part[
        [
            "experiment_id",
            "series_id",
            "rmsse",
            "weight",
            "wrmsse_component"
        ]
    ]

    wrmsse_parts.append(fact_part)

    print(exp_id, fact_part.shape)

fact_wrmsse = pd.concat(
    wrmsse_parts,
    ignore_index=True
)

fact_wrmsse.to_parquet(
    FACT_DIR / "fact_wrmsse.parquet",
    index=False
)

print("fact_wrmsse saved")
print(fact_wrmsse.shape)

fact_wrmsse.head()

exp01_local (30490, 5)
exp02_29f (30490, 5)
exp03_33f (30490, 5)
fact_wrmsse saved
(91470, 5)


,experiment_id,series_id,rmsse,weight,wrmsse_component
0,exp01_local,CA_1_FOODS_1_001,0.862608,0.000017,0.000015
1,exp01_local,CA_1_FOODS_1_002,0.876795,0.000044,0.000039
2,exp01_local,CA_1_FOODS_1_003,0.882911,0.000025,0.000023
3,exp01_local,CA_1_FOODS_1_004,1.780460,0.000165,0.000294
4,exp01_local,CA_1_FOODS_1_005,1.042633,0.000040,0.000041


## 5.5 fact_shap

In [16]:
# =========================
# FACT_SHAP
# =========================

shap_parts = []

for exp_id, cfg in EXPERIMENTS.items():

    exp_path = cfg["path"]

    shap_values = pd.read_parquet(
        exp_path / "shap_values.parquet"
    )

    fact_part = shap_values.copy()
    fact_part["experiment_id"] = exp_id

    if "rank" not in fact_part.columns:
        fact_part["rank"] = (
            fact_part["mean_abs_shap"]
            .rank(
                ascending=False,
                method="dense"
            )
            .astype(int)
        )

    fact_part = fact_part[
        [
            "experiment_id",
            "feature",
            "mean_abs_shap",
            "rank"
        ]
    ]

    shap_parts.append(fact_part)

    print(exp_id, fact_part.shape)

fact_shap = pd.concat(
    shap_parts,
    ignore_index=True
)

fact_shap.to_parquet(
    FACT_DIR / "fact_shap.parquet",
    index=False
)

print("fact_shap saved")
print(fact_shap.shape)

fact_shap.head()

exp01_local (25, 4)
exp02_29f (31, 4)
exp03_33f (35, 4)
fact_shap saved
(91, 4)


,experiment_id,feature,mean_abs_shap,rank
0,exp01_local,rolling_mean_7,0.679284,1
1,exp01_local,rolling_mean_28,0.420694,2
2,exp01_local,lag_1,0.341226,3
3,exp01_local,item_id,0.197346,4
4,exp01_local,snap_WI,0.082957,5


## 5.6 fact_exp_score

In [17]:
# ============================================================
# FACT_EXPERIMENT_SCORE
# Built from execution logs collected during experiments
# ============================================================

import pandas as pd

rows = [
    ("exp01_local", "Local Intel i5 / 32 GB RAM", "local", 25, "Spark feature engineering", 1, 11.94),
    ("exp01_local", "Local Intel i5 / 32 GB RAM", "local", 25, "Model training", 2, 11.87),
    ("exp01_local", "Local Intel i5 / 32 GB RAM", "local", 25, "Post-analysis / predictions", 3, 87.37),
    ("exp01_local", "Local Intel i5 / 32 GB RAM", "local", 25, "Build hierarchy", 4, 2.50),
    ("exp01_local", "Local Intel i5 / 32 GB RAM", "local", 25, "MinT", 5, 1.15),
    ("exp01_local", "Local Intel i5 / 32 GB RAM", "local", 25, "Build M5 evaluation", 6, 1.27),
    ("exp01_local", "Local Intel i5 / 32 GB RAM", "local", 25, "Compute WRMSSE", 7, 0.70),

    ("exp02_29f", "Kaggle CPU", "kaggle", 29, "Spark feature engineering", 1, 15.48),
    ("exp02_29f", "Kaggle CPU", "kaggle", 29, "Model training", 2, 78.17),
    ("exp02_29f", "Kaggle CPU", "kaggle", 29, "Post-analysis / predictions", 3, 181.60),
    ("exp02_29f", "Kaggle CPU", "kaggle", 29, "Build hierarchy", 4, 4.77),
    ("exp02_29f", "Kaggle CPU", "kaggle", 29, "MinT", 5, 2.56),
    ("exp02_29f", "Kaggle CPU", "kaggle", 29, "Build M5 evaluation", 6, 2.98),
    ("exp02_29f", "Kaggle CPU", "kaggle", 29, "Compute WRMSSE", 7, 1.27),
    ("exp02_29f", "Kaggle CPU", "kaggle", 29, "LSTM experimental", 8, 8.12),

    ("exp03_33f", "Kaggle CPU", "kaggle", 33, "Spark feature engineering", 1, 18.15),
    ("exp03_33f", "Kaggle CPU", "kaggle", 33, "Model training", 2, 96.50),
    ("exp03_33f", "Kaggle CPU", "kaggle", 33, "Post-analysis / predictions", 3, 189.09),
    ("exp03_33f", "Kaggle CPU", "kaggle", 33, "Build hierarchy", 4, 4.62),
    ("exp03_33f", "Kaggle CPU", "kaggle", 33, "MinT", 5, 2.16),
    ("exp03_33f", "Kaggle CPU", "kaggle", 33, "Build M5 evaluation", 6, 2.59),
    ("exp03_33f", "Kaggle CPU", "kaggle", 33, "Compute WRMSSE", 7, 1.06),
    ("exp03_33f", "Kaggle CPU", "kaggle", 33, "LSTM experimental", 8, 8.76),
]

fact_timings = pd.DataFrame(
    rows,
    columns=[
        "experiment_id",
        "compute_environment",
        "platform",
        "features_used",
        "stage",
        "stage_order",
        "time_min"
    ]
)

fact_timings["time_hours"] = fact_timings["time_min"] / 60

fact_timings.to_parquet(
    FACT_DIR / "fact_exp_score.parquet",
    index=False
)

print("fact_exp_score saved")
print(fact_timings.shape)

fact_timings.head()

fact_exp_score saved
(23, 8)


,experiment_id,compute_environment,platform,features_used,stage,stage_order,time_min,time_hours
0,exp01_local,Local Intel i5 / 32 GB RAM,local,25,Spark feature engineering,1,11.94,0.199000
1,exp01_local,Local Intel i5 / 32 GB RAM,local,25,Model training,2,11.87,0.197833
2,exp01_local,Local Intel i5 / 32 GB RAM,local,25,Post-analysis / predictions,3,87.37,1.456167
3,exp01_local,Local Intel i5 / 32 GB RAM,local,25,Build hierarchy,4,2.50,0.041667
4,exp01_local,Local Intel i5 / 32 GB RAM,local,25,MinT,5,1.15,0.019167


# 6. export_zip

## 6.1 Verificación previa a zip

In [18]:
# =========================
# VERIFY HIERARCHY SHAPES
# =========================

for exp_id, cfg in EXPERIMENTS.items():

    exp_path = cfg["path"]

    h = pd.read_parquet(
        exp_path / "hierarchy_predictions.parquet",
        columns=["series_id", "date", "level"]
    )

    summary = (
        h.groupby("level")
        .agg(
            rows=("series_id", "size"),
            series=("series_id", "nunique"),
            dates=("date", "nunique")
        )
        .reset_index()
        .sort_values("level")
    )

    print("\n" + "=" * 60)
    print(exp_id)
    print(summary)


exp01_local
        level      rows  series  dates
0         cat     57390      30   1913
1        dept    133910      70   1913
2        item  58327370   30490   1913
3       state     19130       3   1913
4   state_cat     57390       9   1913
5  state_dept    133910      21   1913
6       store     19130      10   1913
7       total     19130       1   1913

exp02_29f
        level      rows  series  dates
0         cat     57390      30   1913
1        dept    133910      70   1913
2        item  58327370   30490   1913
3       state      5739       3   1913
4   state_cat     17217       9   1913
5  state_dept     40173      21   1913
6       store     19130      10   1913
7       total      1913       1   1913

exp03_33f
        level      rows  series  dates
0         cat     57390      30   1913
1        dept    133910      70   1913
2        item  58327370   30490   1913
3       state      5739       3   1913
4   state_cat     17217       9   1913
5  state_dept     40173      

In [19]:
forecast_agg = pd.read_parquet(FACT_DIR / "fact_forecast_agg")

check_fact = (
    forecast_agg
    .groupby(["experiment_id", "level"], as_index=False)
    .agg(
        rows=("series_id", "size"),
        series=("series_id", "nunique"),
        dates=("date", "nunique")
    )
)

check_fact

/tmp/ipykernel_219/2948976151.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["experiment_id", "level"], as_index=False)


,experiment_id,level,rows,series,dates
0,exp01_local,cat,57390,30,1913
1,exp01_local,dept,133910,70,1913
2,exp01_local,state,5739,3,1913
3,exp01_local,state_cat,17217,9,1913
4,exp01_local,state_dept,40173,21,1913
5,exp01_local,store,19130,10,1913
6,exp01_local,total,1913,1,1913
7,exp02_29f,cat,57390,30,1913
8,exp02_29f,dept,133910,70,1913
9,exp02_29f,state,5739,3,1913


In [20]:
print("Dims:")
for p in sorted(DIM_DIR.glob("*")):
    print(p.name)

print("\nFacts:")
for p in sorted(FACT_DIR.glob("*")):
    print(p.name)

Dims:
dim_date.parquet
dim_experiment.parquet
dim_feature.parquet
dim_level.parquet
dim_series.parquet

Facts:
fact_exp_score.parquet
fact_forecast_agg
fact_forecast_monthly.parquet
fact_shap.parquet
fact_top_items.parquet
fact_wrmsse.parquet


## 6.2 exportar zip

In [21]:
%cd /kaggle/working

!zip -r TFM_powerbi_lake.zip TFM \
-x "*.ipynb_checkpoints*" \
-x "__pycache__/*"

/kaggle/working
updating: TFM/ (stored 0%)
updating: TFM/powerbi/ (stored 0%)
updating: TFM/powerbi/tools/ (stored 0%)
updating: TFM/powerbi/tools/create_powerbi_model.sql (deflated 81%)
updating: TFM/powerbi/tools/duckdb/ (stored 0%)
updating: TFM/powerbi/tools/duckdb_odbc/ (stored 0%)
updating: TFM/powerbi/data/ (stored 0%)
updating: TFM/powerbi/data/powerbi_lake/ (stored 0%)
updating: TFM/powerbi/data/powerbi_lake/facts/ (stored 0%)
updating: TFM/powerbi/data/powerbi_lake/facts/fact_top_items.parquet (deflated 13%)
updating: TFM/powerbi/data/powerbi_lake/facts/fact_forecast_agg/ (stored 0%)
updating: TFM/powerbi/data/powerbi_lake/facts/fact_forecast_agg/experiment_id=exp01_local/ (stored 0%)
updating: TFM/powerbi/data/powerbi_lake/facts/fact_forecast_agg/experiment_id=exp01_local/level=store/ (stored 0%)
updating: TFM/powerbi/data/powerbi_lake/facts/fact_forecast_agg/experiment_id=exp01_local/level=dept/ (stored 0%)
updating: TFM/powerbi/data/powerbi_lake/facts/fact_forecast_agg/exp

In [22]:
# crear enlace para la descarga
from IPython.display import FileLink
FileLink(r'TFM_powerbi_lake.zip')

/kaggle/working/TFM_powerbi_lake.zip